# Step 5 — Model culture instead of erasing it: Consensus–Deviation Decomposition, Group DRO, per-culture thresholds

Steps 3–3c showed that removing culture (GRL or provable LEACE erasure) does not significantly reduce cross-cultural disparities, because Multi3Hate labels themselves differ by culture. This notebook tests methods that **model** that difference.

**Consensus–Deviation Decomposition (CDD)** — for meme *m* shown to culture *k*:
`logit(m, k) = s(m) + δ_k(m)`
- `s(m)` = cross-cultural **consensus** head, additionally trained on the mean vote across all five cultures of the meme (available because every meme is annotated by all cultures)
- `δ_k(m)` = **culture-specific deviation** head (content + culture embedding), L2-shrunk towards 0 so deviations are used only when the data require them

**Group DRO** (Sagawa et al., 2020) — minimises the worst loss over the 10 culture × label groups.

**Per-culture thresholds** — post-processing baseline applied to *every* model (reported as `+pcthr`).

| run | head | soft labels | LEACE | Group DRO |
|---|---|---|---|---|
| `plain` | plain | – | – | – |
| `cond_soft` | culture FiLM | ✓ | – | – |
| `cond_soft_dro` | culture FiLM | ✓ | – | ✓ |
| `cdd` | CDD | ✓ | – | – |
| `leace_cdd` | CDD | ✓ | ✓ | – |
| `leace_cdd_dro` | CDD | ✓ | ✓ | ✓ |

Same 5×5 meme-grouped CV as before. Also reports **disagreement alignment**: does the model's predicted cross-cultural disagreement per meme match the real disagreement between annotator groups?

**Kaggle:** GPU T4, Internet On, **Save Version → Save & Run All** (~2 h). Finished runs are skipped on re-run.

In [1]:
import glob, json, math, os, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from types import SimpleNamespace
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.neural_network import MLPClassifier
import warnings; warnings.filterwarnings("ignore", category=UserWarning)

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/kaggle/working/**/{name}", recursive=True)
    return hits[0] if hits else None

FEATURES = {"openclip_b32": find("feats_openclip_b32.npz"), "siglip2": find("feats_siglip2.npz")}
print("Attached inputs:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "none")
print("Found features :", FEATURES)
if not all(FEATURES.values()):
    print("\nFeature files not found as input -> building them here (needs GPU + Internet On, ~5 min).")
RESULTS = "/kaggle/working/results_step5"
os.makedirs(RESULTS, exist_ok=True)

Attached inputs: []
Found features : {'openclip_b32': None, 'siglip2': None}

Feature files not found as input -> building them here (needs GPU + Internet On, ~5 min).


In [2]:
# ---- Build features in this notebook if they were not attached as input (same code as Step 2) ----
if not all(FEATURES.values()):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.49", "open_clip_torch"], check=True)
    from PIL import Image

    REPO = "/kaggle/working/Multi3Hate"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/MinhDucBui/Multi3Hate.git", REPO], check=True)
    DATA = f"{REPO}/data"
    LANG2CULT = {"en": "US", "de": "DE", "es": "MX", "hi": "IN", "zh": "CN"}
    final = pd.read_csv(f"{DATA}/final_annotations.csv")
    raw = pd.read_csv(f"{DATA}/raw_annotations.csv").dropna(subset=["hatespeech"])
    votes = raw.groupby(["dataset_language", "Meme ID"])["hatespeech"].agg(n_hate="sum", n_total="count")
    img_index = {(p.split("/")[-3], int(os.path.splitext(os.path.basename(p))[0])): p
                 for p in glob.glob(f"{DATA}/memes/*/*/*")}
    rows = []
    for lang, cult in LANG2CULT.items():
        for _, r in pd.read_csv(f"{DATA}/captions/{lang}.csv").iterrows():
            mid = int(r["Meme ID"])
            v = votes.loc[(lang, mid)]
            rows.append({"meme_id": mid, "culture": cult, "language": lang,
                         "image_path": img_index[(lang, mid)],
                         "text": str(r["Translation"]).replace("<sep>", " / "),
                         "n_hate": int(v["n_hate"]), "n_total": int(v["n_total"]),
                         "label": int(final.loc[final["Meme ID"] == mid, cult].iloc[0])})
    df = pd.DataFrame(rows).sort_values(["meme_id", "culture"]).reset_index(drop=True)
    df.to_csv("/kaggle/working/multi3hate.csv", index=False)
    print("table:", df.shape)

    DEVICE_FE = "cuda" if torch.cuda.is_available() else "cpu"

    def as_tensor(out):
        if isinstance(out, torch.Tensor):
            return out
        for key in ("pooler_output", "image_embeds", "text_embeds"):
            v = getattr(out, key, None)
            if isinstance(v, torch.Tensor):
                return v
        raise TypeError(type(out))

    def load_backend(name):
        if name == "siglip2":
            from transformers import AutoModel, AutoProcessor
            mid = "google/siglip2-base-patch16-224"
            model = AutoModel.from_pretrained(mid).to(DEVICE_FE).eval()
            proc = AutoProcessor.from_pretrained(mid)
            enc_img = lambda ims: as_tensor(model.get_image_features(**proc(images=ims, return_tensors="pt").to(DEVICE_FE)))
            enc_txt = lambda ts: as_tensor(model.get_text_features(**proc(
                text=[t.lower() for t in ts], padding="max_length", max_length=64,
                truncation=True, return_tensors="pt").to(DEVICE_FE)))
            return enc_img, enc_txt
        import open_clip
        model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
        model = model.to(DEVICE_FE).eval()
        tok = open_clip.get_tokenizer("ViT-B-32")
        enc_img = lambda ims: model.encode_image(torch.stack([preprocess(i) for i in ims]).to(DEVICE_FE))
        enc_txt = lambda ts: model.encode_text(tok(ts).to(DEVICE_FE))
        return enc_img, enc_txt

    @torch.no_grad()
    def extract(backend, out_path, batch=32):
        enc_img, enc_txt = load_backend(backend)
        img, txt = [], []
        for i in range(0, len(df), batch):
            chunk = df.iloc[i:i + batch]
            ims = [Image.open(p).convert("RGB") for p in chunk.image_path]
            img.append(F.normalize(enc_img(ims).float(), dim=-1).cpu().numpy())
            txt.append(F.normalize(enc_txt(chunk.text.tolist()).float(), dim=-1).cpu().numpy())
        img, txt = np.concatenate(img), np.concatenate(txt)
        np.savez(out_path, X=np.concatenate([img, txt], 1).astype(np.float32), img_dim=img.shape[1],
                 culture=df.culture.values.astype(str), meme_id=df.meme_id.values.astype(str),
                 y_soft=(df.n_hate / df.n_total).values.astype(np.float32),
                 y_hard=df.label.values.astype(np.int64), backend=backend)
        print(f"saved {out_path}: {img.shape[0]} x {img.shape[1] + txt.shape[1]}")
        torch.cuda.empty_cache()

    for bk, backend in [("openclip_b32", "openclip"), ("siglip2", "siglip2")]:
        if not FEATURES[bk]:
            path = f"/kaggle/working/feats_{bk}.npz"
            extract(backend, path)
            FEATURES[bk] = path
    import shutil; shutil.rmtree(REPO, ignore_errors=True)

print("Using features:", FEATURES)
assert all(FEATURES.values())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 872.5 kB/s eta 0:00:00


Cloning into '/kaggle/working/Multi3Hate'...


table: (1500, 8)


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

saved /kaggle/working/feats_openclip_b32.npz: 1500 x 1024


...

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

saved /kaggle/working/feats_siglip2.npz: 1500 x 1536
Using features: {'openclip_b32': '/kaggle/working/feats_openclip_b32.npz', 'siglip2': '/kaggle/working/feats_siglip2.npz'}


In [3]:
# ---- Shared evaluation metrics ----
import numpy as np
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix)

CULTURES = ["US", "DE", "MX", "IN", "CN"]


def _f(x):
    """Plain float for JSON; NaN -> None."""
    if x is None:
        return None
    x = float(x)
    return None if np.isnan(x) else x


def safe_auc(y, p):
    return roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan


def safe_bal_acc(y, yhat):
    return balanced_accuracy_score(y, yhat) if len(np.unique(y)) == 2 else np.nan


def expected_calibration_error(y, p, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(y[m].mean() - p[m].mean())
    return ece


def bernoulli_jsd(q, p, eps=1e-6):
    """Jensen-Shannon divergence (bits) between Bernoulli(q) and Bernoulli(p)."""
    q = np.clip(q, eps, 1 - eps)
    p = np.clip(p, eps, 1 - eps)
    m = 0.5 * (q + p)

    def kl(a, b):
        return a * np.log2(a / b) + (1 - a) * np.log2((1 - a) / (1 - b))

    return 0.5 * kl(q, m) + 0.5 * kl(p, m)


def _spread(vals):
    v = np.array([x for x in vals if x is not None and not np.isnan(x)], dtype=float)
    return (v.max() - v.min()) if len(v) else np.nan


def evaluate(y_true, y_prob, culture, y_soft=None, thr=0.5):
    """Full metric set: overall, per-culture, fairness gaps, calibration, soft-label fit."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    culture = np.asarray(culture).astype(str)
    y_pred = (y_prob >= thr).astype(int)

    out = {
        "n": int(len(y_true)),
        "threshold": _f(np.mean(thr)),
        "pos_rate_true": _f(y_true.mean()),
        "pos_rate_pred": _f(y_pred.mean()),
        "accuracy": _f(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": _f(safe_bal_acc(y_true, y_pred)),
        "macro_f1": _f(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_hate": _f(f1_score(y_true, y_pred, zero_division=0)),
        "auroc": _f(safe_auc(y_true, y_prob)),
        "ece": _f(expected_calibration_error(y_true, y_prob)),
        "constant_all_hate_acc": _f(y_true.mean()),  # reference only
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }
    out["degenerate"] = bool(out["pos_rate_pred"] > 0.95 or out["pos_rate_pred"] < 0.05)

    per = {}
    for c in [k for k in CULTURES if k in set(culture)] + sorted(set(culture) - set(CULTURES)):
        m = culture == c
        yt, yp = y_true[m], y_pred[m]
        per[c] = {
            "n": int(m.sum()),
            "hate_prevalence": _f(yt.mean()),
            "accuracy": _f((yt == yp).mean()),
            "balanced_accuracy": _f(safe_bal_acc(yt, yp)),
            "tpr": _f(yp[yt == 1].mean()) if (yt == 1).any() else None,
            "fpr": _f(yp[yt == 0].mean()) if (yt == 0).any() else None,
            "pos_rate_pred": _f(yp.mean()),
        }
    out["per_culture"] = per

    def col(key):
        return [np.nan if v[key] is None else v[key] for v in per.values()]

    accs, bals = col("accuracy"), col("balanced_accuracy")
    out["worst_culture_acc"] = _f(np.nanmin(accs))
    out["worst_culture_bal_acc"] = _f(np.nanmin(bals)) if not np.all(np.isnan(bals)) else None
    out["bias_gap_acc"] = _f(_spread(accs))
    out["bias_gap_bal_acc"] = _f(_spread(bals))
    out["tpr_gap"] = _f(_spread(col("tpr")))
    out["fpr_gap"] = _f(_spread(col("fpr")))
    gaps = [g for g in (out["tpr_gap"], out["fpr_gap"]) if g is not None]
    out["equalized_odds_gap"] = _f(max(gaps)) if gaps else None
    # What a constant "hate" predictor would score as a bias gap on this split:
    out["constant_predictor_bias_gap"] = _f(_spread(col("hate_prevalence")))

    if y_soft is not None:
        q = np.asarray(y_soft, dtype=float)
        p = np.clip(y_prob, 1e-6, 1 - 1e-6)
        out["soft_bce"] = _f(np.mean(-(q * np.log(p) + (1 - q) * np.log(1 - p))))
        out["soft_jsd"] = _f(np.mean(bernoulli_jsd(q, y_prob)))
    return out

In [4]:
# ---- LEACE: closed-form least-squares concept erasure (Belrose et al., 2023) ----
def fit_leace(X, c, n_cls, eps=1e-6):
    """Returns (mu, E) such that x' = x - (x - mu) @ E.T removes all linear information about c."""
    X = X.astype(np.float64)
    mu = X.mean(0)
    Xc = X - mu
    Z = np.eye(n_cls)[c]
    Zc = Z - Z.mean(0)
    n = len(X)
    Sxx = Xc.T @ Xc / n
    Sxz = Xc.T @ Zc / n
    evals, evecs = np.linalg.eigh(Sxx)
    keep = evals > eps * evals.max()
    V, ev = evecs[:, keep], evals[keep]
    W = V @ np.diag(ev ** -0.5) @ V.T          # whitening
    W_pinv = V @ np.diag(ev ** 0.5) @ V.T      # its pseudo-inverse
    U, s, _ = np.linalg.svd(W @ Sxz, full_matrices=False)
    U = U[:, s > 1e-8 * s.max()]
    E = W_pinv @ (U @ U.T) @ W
    return mu, E

def apply_leace(X, mu, E):
    return (X - (X - mu) @ E.T).astype(np.float32)

In [5]:
# ---- Model, training, probes, baselines ----
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return -ctx.lam * g, None


class CCAL(nn.Module):
    def __init__(self, in_dim, hid, n_cult, cult_dim, p, adv, conditioned):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(hid, hid), nn.ReLU(), nn.Dropout(p))
        self.conditioned = conditioned
        if conditioned:
            self.cult_emb = nn.Embedding(n_cult, cult_dim)
            self.film = nn.Linear(cult_dim, 2 * hid)
            nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)  # start as identity
        self.head = nn.Sequential(nn.Linear(hid, 128), nn.ReLU(), nn.Dropout(p), nn.Linear(128, 1))
        self.adv_type = adv
        if adv != "none":
            extra = 2 if adv == "cond_grl" else 0
            self.adv = nn.Sequential(nn.Linear(hid + extra, 128), nn.ReLU(), nn.Dropout(p),
                                     nn.Linear(128, n_cult))

    def forward(self, x, c, y_hard=None, lam=0.0, with_adv=False):
        z = self.enc(x)
        h = z
        if self.conditioned:
            g, b = self.film(self.cult_emb(c)).chunk(2, dim=-1)
            h = z * (1 + g) + b
        logit = self.head(h).squeeze(-1)
        adv_logit = None
        if with_adv and self.adv_type != "none":
            zr = GradReverse.apply(z, lam)
            if self.adv_type == "cond_grl":
                zr = torch.cat([zr, F.one_hot(y_hard, 2).float()], dim=-1)
            adv_logit = self.adv(zr)
        return logit, adv_logit, z


def lam_at(epoch, total, lam_max):
    p = epoch / max(total - 1, 1)
    return lam_max * (2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)


@torch.no_grad()
def predict(model, X, C, bs=512):
    model.eval()
    probs, zs = [], []
    for i in range(0, len(X), bs):
        x = torch.as_tensor(X[i:i + bs], dtype=torch.float32, device=DEV)
        c = torch.as_tensor(C[i:i + bs], dtype=torch.long, device=DEV)
        logit, _, z = model(x, c)
        probs.append(torch.sigmoid(logit).cpu().numpy()); zs.append(z.cpu().numpy())
    return np.concatenate(probs), np.concatenate(zs)


def tune_threshold(y, p):
    grid = np.linspace(0.05, 0.95, 91)
    scores = [balanced_accuracy_score(y, (p >= t).astype(int)) for t in grid]
    return float(grid[int(np.argmax(scores))])


def train_one(d, tr, va, lam_max, a, seed):
    set_seed(seed)
    X, C, YH, YS = d["X"], d["c"], d["yh"], d["ys"]
    model = CCAL(X.shape[1], a.hidden, len(CULTURES), a.cult_dim, a.dropout,
                 a.adv, not a.no_condition).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    target = YS if a.soft else YH.astype(np.float32)
    n_pos = YH[tr].sum()
    pos_weight = torch.tensor((len(tr) - n_pos) / max(n_pos, 1), dtype=torch.float32, device=DEV)

    ds = TensorDataset(torch.tensor(X[tr], dtype=torch.float32), torch.tensor(C[tr]),
                       torch.tensor(YH[tr]), torch.tensor(target[tr], dtype=torch.float32))
    dl = DataLoader(ds, batch_size=a.batch, shuffle=True,
                    generator=torch.Generator().manual_seed(seed))

    best, best_state, best_thr, bad = -np.inf, None, 0.5, 0
    for ep in range(a.epochs):
        lam = lam_at(ep, a.epochs, lam_max) if a.adv != "none" else 0.0
        model.train()
        for xb, cb, yhb, tb in dl:
            xb, cb, yhb, tb = xb.to(DEV), cb.to(DEV), yhb.to(DEV), tb.to(DEV)
            logit, adv_logit, _ = model(xb, cb, yhb, lam, with_adv=a.adv != "none")
            loss = F.binary_cross_entropy_with_logits(logit, tb, pos_weight=pos_weight)
            if adv_logit is not None:
                # DANN convention: adversary trained at full strength,
                # encoder receives -lambda * gradient through the GRL.
                loss = loss + F.cross_entropy(adv_logit, cb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # Model selection starts only after min_epochs, so the chosen checkpoint
        # has the adversary substantially ramped up (not an early, un-debiased one).
        if ep < a.min_epochs and ep < a.epochs - 1:
            continue
        p_va, _ = predict(model, X[va], C[va])
        thr = tune_threshold(YH[va], p_va)
        score = evaluate(YH[va], p_va, d["cname"][va], YS[va], thr)[a.select]
        score = -np.inf if score is None else score
        if best_state is None or score > best + 1e-4:
            best, best_thr, bad = score, thr, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= a.patience:
                break
    model.load_state_dict(best_state)
    return model, best, best_thr


def leakage_probes(model, d, tr, te, seed):
    _, ztr = predict(model, d["X"][tr], d["c"][tr])
    _, zte = predict(model, d["X"][te], d["c"][te])
    lin = LogisticRegression(max_iter=3000).fit(ztr, d["c"][tr]).score(zte, d["c"][te])
    mlp = MLPClassifier((128,), max_iter=600, random_state=seed).fit(ztr, d["c"][tr]).score(zte, d["c"][te])
    raw = LogisticRegression(max_iter=3000).fit(d["X"][tr], d["c"][tr]).score(d["X"][te], d["c"][te])
    return {"probe_z_linear_acc": float(lin), "probe_z_mlp_acc": float(mlp),
            "probe_input_linear_acc": float(raw), "probe_chance": 1 / len(CULTURES)}


def baselines(d, tr, te):
    yh, c = d["yh"], d["c"]
    const = np.full(len(te), float(yh[tr].mean() >= 0.5))
    per = np.array([yh[tr][c[tr] == k].mean() for k in range(len(CULTURES))])
    return {
        "majority": evaluate(yh[te], const, d["cname"][te], d["ys"][te]),
        "per_culture_majority": evaluate(yh[te], per[c[te]], d["cname"][te], d["ys"][te]),
    }


def load(path, modality):
    z = np.load(path, allow_pickle=True)
    X = z["X"].astype(np.float32)
    k = int(z["img_dim"])
    X = {"both": X, "image": X[:, :k], "text": X[:, k:]}[modality]
    cname = z["culture"].astype(str)
    return dict(X=X, cname=cname, c=np.array([CULTURES.index(s) for s in cname]),
                meme=z["meme_id"].astype(str), ys=z["y_soft"].astype(np.float32),
                yh=z["y_hard"].astype(np.int64))


SUMMARY_KEYS = ["accuracy", "balanced_accuracy", "macro_f1", "auroc", "ece",
                "worst_culture_acc", "worst_culture_bal_acc", "bias_gap_acc",
                "bias_gap_bal_acc", "equalized_odds_gap", "pos_rate_pred",
                "soft_bce", "soft_jsd", "probe_z_linear_acc", "probe_z_mlp_acc",
                "probe_input_linear_acc"]


def summarize(rows):
    s = {}
    for k in SUMMARY_KEYS:
        v = np.array([r[k] for r in rows if r.get(k) is not None], dtype=float)
        if len(v):
            s[k] = {"mean": float(v.mean()), "std": float(v.std(ddof=1)) if len(v) > 1 else 0.0}
    s["degenerate_folds"] = int(sum(r["degenerate"] for r in rows))
    s["n_folds"] = len(rows)
    return s

In [6]:
# ---- Step 5 model: plain / culture-FiLM / Consensus-Deviation Decomposition ----
class Net(nn.Module):
    def __init__(self, in_dim, hid, n_cult, cult_dim, p, head):
        super().__init__()
        self.head_type = head
        self.enc = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(hid, hid), nn.ReLU(), nn.Dropout(p))
        self.cls = nn.Sequential(nn.Linear(hid, 128), nn.ReLU(), nn.Dropout(p), nn.Linear(128, 1))
        if head in ("film", "cdd"):
            self.emb = nn.Embedding(n_cult, cult_dim)
        if head == "film":
            self.film = nn.Linear(cult_dim, 2 * hid)
            nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)
        if head == "cdd":
            self.dev = nn.Sequential(nn.Linear(hid + cult_dim, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 1))
            nn.init.zeros_(self.dev[-1].weight); nn.init.zeros_(self.dev[-1].bias)  # start at consensus

    def forward(self, x, c):
        z = self.enc(x)
        if self.head_type == "plain":
            return self.cls(z).squeeze(-1), None, z
        if self.head_type == "film":
            g, b = self.film(self.emb(c)).chunk(2, dim=-1)
            return self.cls(z * (1 + g) + b).squeeze(-1), None, z
        cons = self.cls(z).squeeze(-1)                                    # consensus s(m)
        dev = self.dev(torch.cat([z, self.emb(c)], dim=-1)).squeeze(-1)   # deviation delta_k(m)
        return cons + dev, (cons, dev), z


def consensus_targets(d):
    """Mean soft label of each meme across its five cultures (training rows only are ever used)."""
    return pd.Series(d["ys"]).groupby(d["meme"]).transform("mean").values.astype(np.float32)


def tune_threshold_per_culture(y, p, c, default):
    thr = np.full(len(CULTURES), default, dtype=float)
    for k in range(len(CULTURES)):
        m = c == k
        if m.sum() >= 6 and len(np.unique(y[m])) == 2:
            thr[k] = tune_threshold(y[m], p[m])
    return thr


@torch.no_grad()
def predict_parts(model, X, C):
    model.eval()
    _, aux, _ = model(torch.as_tensor(X, dtype=torch.float32, device=DEV),
                      torch.as_tensor(C, dtype=torch.long, device=DEV))
    if aux is None:
        return np.full(len(X), np.nan), np.full(len(X), np.nan)
    return torch.sigmoid(aux[0]).cpu().numpy(), aux[1].cpu().numpy()


def train_net(d, tr, va, a, seed):
    set_seed(seed)
    X, C, YH, YS = d["X"], d["c"], d["yh"], d["ys"]
    model = Net(X.shape[1], a.hidden, len(CULTURES), a.cult_dim, a.dropout, a.head).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    target = YS if a.soft else YH.astype(np.float32)
    n_pos = YH[tr].sum()
    pw = torch.tensor((len(tr) - n_pos) / max(n_pos, 1), dtype=torch.float32, device=DEV)
    G = C * 2 + YH
    q = torch.ones(2 * len(CULTURES), device=DEV) / (2 * len(CULTURES))
    ds = TensorDataset(torch.tensor(X[tr], dtype=torch.float32), torch.tensor(C[tr]),
                       torch.tensor(target[tr], dtype=torch.float32),
                       torch.tensor(d["cons"][tr]), torch.tensor(G[tr]))
    dl = DataLoader(ds, batch_size=a.batch, shuffle=True, generator=torch.Generator().manual_seed(seed))
    best, best_state, bad = -np.inf, None, 0
    for ep in range(a.epochs):
        model.train()
        for xb, cb, tb, consb, gb in dl:
            xb, cb, tb, consb, gb = xb.to(DEV), cb.to(DEV), tb.to(DEV), consb.to(DEV), gb.to(DEV)
            logit, aux, _ = model(xb, cb)
            per = F.binary_cross_entropy_with_logits(logit, tb, pos_weight=pw, reduction="none")
            if a.dro:   # online Group DRO over culture x label groups
                groups = gb.unique()
                gl = torch.stack([per[gb == g].mean() for g in groups])
                with torch.no_grad():
                    q[groups] *= torch.exp(a.dro_eta * gl.detach())
                    q /= q.sum()
                w = q[groups]
                loss = (w * gl).sum() / w.sum()
            else:
                loss = per.mean()
            if aux is not None:
                cons, dev = aux
                loss = loss + a.beta * F.binary_cross_entropy_with_logits(cons, consb) + a.gamma * (dev ** 2).mean()
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        if ep < a.min_epochs and ep < a.epochs - 1:
            continue
        p_va, _ = predict(model, X[va], C[va])
        thr = tune_threshold(YH[va], p_va)
        score = evaluate(YH[va], p_va, d["cname"][va], YS[va], thr)[a.select]
        score = -np.inf if score is None else score
        if best_state is None or score > best + 1e-4:
            best, bad = score, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= a.patience:
                break
    model.load_state_dict(best_state)
    return model

In [7]:
DEFAULTS = dict(head="plain", soft=False, dro=False, erase=False, beta=0.5, gamma=0.05, dro_eta=0.05,
                modality="both", select="balanced_accuracy", folds=5, repeats=5, epochs=60,
                min_epochs=10, patience=10, batch=32, hidden=256, cult_dim=32, dropout=0.3,
                lr=1e-4, wd=1e-4, seed=42)

def run_experiment(name, features, **overrides):
    out = os.path.join(RESULTS, name)
    if os.path.exists(os.path.join(out, "summary.json")):
        print(f"[skip] {name} already done")
        return json.load(open(os.path.join(out, "summary.json")))
    a = SimpleNamespace(**{**DEFAULTS, **overrides})
    os.makedirs(out, exist_ok=True)
    d0 = load(features, a.modality)
    d0["cons"] = consensus_targets(d0)
    rows_g, rows_pc, base_rows, preds = [], [], {"majority": [], "per_culture_majority": []}, []
    t0 = time.time()
    print(f"\n===== {name} =====")
    for rep in range(a.repeats):
        sgkf = StratifiedGroupKFold(n_splits=a.folds, shuffle=True, random_state=1000 + rep)
        for fold, (tr_all, te) in enumerate(sgkf.split(d0["X"], d0["yh"], groups=d0["meme"])):
            seed = a.seed + 100 * rep + fold
            gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
            itr, iva = next(gss.split(tr_all, d0["yh"][tr_all], groups=d0["meme"][tr_all]))
            tr, va = tr_all[itr], tr_all[iva]
            d = d0
            if a.erase:
                mu, E = fit_leace(d0["X"][tr_all], d0["c"][tr_all], len(CULTURES))
                d = dict(d0, X=apply_leace(d0["X"], mu, E))
            model = train_net(d, tr, va, a, seed)
            p_va, _ = predict(model, d["X"][va], d["c"][va])
            thr_g = tune_threshold(d["yh"][va], p_va)
            thr_pc = tune_threshold_per_culture(d["yh"][va], p_va, d["c"][va], thr_g)
            p_te, _ = predict(model, d["X"][te], d["c"][te])
            cons_te, dev_te = predict_parts(model, d["X"][te], d["c"][te])
            probes = leakage_probes(model, d, tr, te, seed)
            info = {"repeat": rep, "fold": fold}
            m_g = evaluate(d["yh"][te], p_te, d["cname"][te], d["ys"][te], thr_g); m_g.update(probes); m_g.update(info)
            m_pc = evaluate(d["yh"][te], p_te, d["cname"][te], d["ys"][te], thr_pc[d["c"][te]]); m_pc.update(probes); m_pc.update(info)
            rows_g.append(m_g); rows_pc.append(m_pc)
            for k, v in baselines(d, tr_all, te).items():
                base_rows[k].append(v)
            preds.append(pd.DataFrame({
                "repeat": rep, "fold": fold, "meme_id": d["meme"][te], "culture": d["cname"][te],
                "y_true": d["yh"][te], "y_soft": d["ys"][te], "y_prob": p_te,
                "y_pred": (p_te >= thr_g).astype(int), "y_pred_pcthr": (p_te >= thr_pc[d["c"][te]]).astype(int),
                "consensus_prob": cons_te, "deviation_logit": dev_te}))
            f3 = lambda v: "nan" if v is None else f"{v:.3f}"
            print(f"rep {rep} fold {fold} | bal_acc={f3(m_g['balanced_accuracy'])} auroc={f3(m_g['auroc'])} "
                  f"worst={f3(m_g['worst_culture_bal_acc'])} eo_gap={f3(m_g['equalized_odds_gap'])} "
                  f"| +pcthr worst={f3(m_pc['worst_culture_bal_acc'])} eo_gap={f3(m_pc['equalized_odds_gap'])}"
                  f"{'  <-- DEGENERATE' if m_g['degenerate'] else ''}")
    summary = {"name": name, "features": os.path.basename(features), "config": vars(a),
               "model": summarize(rows_g), "model_pcthr": summarize(rows_pc),
               "baselines": {k: summarize(v) for k, v in base_rows.items()},
               "minutes": round((time.time() - t0) / 60, 1)}
    json.dump({"model": rows_g, "model_pcthr": rows_pc, "baselines": base_rows},
              open(os.path.join(out, "fold_metrics.json"), "w"), indent=1)
    pd.concat(preds).to_csv(os.path.join(out, "predictions.csv"), index=False)
    json.dump(summary, open(os.path.join(out, "summary.json"), "w"), indent=1)
    s = summary["model"]
    print(f"--> {name}: bal_acc {s['balanced_accuracy']['mean']:.3f}±{s['balanced_accuracy']['std']:.3f} | "
          f"worst {s['worst_culture_bal_acc']['mean']:.3f} | eo_gap {s['equalized_odds_gap']['mean']:.3f} | "
          f"degenerate {s['degenerate_folds']}/{s['n_folds']} | {summary['minutes']} min")
    return summary

### Quick test (optional)
Set `QUICK_TEST = True` to check everything runs in a few minutes, then set it back to `False` for the real run.

In [8]:
QUICK_TEST = False
if QUICK_TEST:
    run_experiment("_quicktest", FEATURES["siglip2"], head="cdd", soft=True, erase=True, dro=True,
                   folds=3, repeats=1, epochs=15, min_epochs=3)
    import shutil; shutil.rmtree(os.path.join(RESULTS, "_quicktest"), ignore_errors=True)

### Full experiment matrix

In [9]:
VARIANTS = [
    ("plain",         dict(head="plain")),
    ("cond_soft",     dict(head="film", soft=True)),
    ("cond_soft_dro", dict(head="film", soft=True, dro=True)),
    ("cdd",           dict(head="cdd",  soft=True)),
    ("leace_cdd",     dict(head="cdd",  soft=True, erase=True)),
    ("leace_cdd_dro", dict(head="cdd",  soft=True, erase=True, dro=True)),
]
if not QUICK_TEST:
    for bk in ["siglip2", "openclip_b32"]:
        for tag, cfg in VARIANTS:
            run_experiment(f"{bk}_{tag}", FEATURES[bk], **cfg)


===== siglip2_plain =====
rep 0 fold 0 | bal_acc=0.649 auroc=0.755 worst=0.575 eo_gap=0.367 | +pcthr worst=0.593 eo_gap=0.367
rep 0 fold 1 | bal_acc=0.613 auroc=0.668 worst=0.558 eo_gap=0.199 | +pcthr worst=0.546 eo_gap=0.326
rep 0 fold 2 | bal_acc=0.658 auroc=0.731 worst=0.606 eo_gap=0.226 | +pcthr worst=0.560 eo_gap=0.880
rep 0 fold 3 | bal_acc=0.553 auroc=0.609 worst=0.505 eo_gap=0.243 | +pcthr worst=0.562 eo_gap=0.541
rep 0 fold 4 | bal_acc=0.626 auroc=0.670 worst=0.589 eo_gap=0.169 | +pcthr worst=0.574 eo_gap=0.366
rep 1 fold 0 | bal_acc=0.663 auroc=0.739 worst=0.617 eo_gap=0.147 | +pcthr worst=0.617 eo_gap=0.258
rep 1 fold 1 | bal_acc=0.663 auroc=0.699 worst=0.524 eo_gap=0.311 | +pcthr worst=0.524 eo_gap=0.365
rep 1 fold 2 | bal_acc=0.646 auroc=0.694 worst=0.583 eo_gap=0.179 | +pcthr worst=0.601 eo_gap=0.212
rep 1 fold 3 | bal_acc=0.694 auroc=0.758 worst=0.578 eo_gap=0.224 | +pcthr worst=0.596 eo_gap=0.422
rep 1 fold 4 | bal_acc=0.603 auroc=0.630 worst=0.528 eo_gap=0.214 | +pcth

### Results table (global threshold and per-culture thresholds)

In [10]:
KEYS = ["balanced_accuracy", "auroc", "macro_f1", "worst_culture_bal_acc", "bias_gap_bal_acc",
        "equalized_odds_gap", "tpr_gap", "fpr_gap", "ece", "soft_jsd", "probe_z_mlp_acc"]
SUMMARY_KEYS[:] = sorted(set(SUMMARY_KEYS) | set(KEYS))
rows = []
for f in sorted(glob.glob(f"{RESULTS}/*/summary.json")):
    s = json.load(open(f))
    for part, suffix in [("model", ""), ("model_pcthr", " +pcthr")]:
        # recompute summaries so that tpr_gap / fpr_gap are included
        fm = json.load(open(f.replace("summary.json", "fold_metrics.json")))[part]
        sm = summarize(fm)
        r = {"run": s["name"] + suffix}
        for k in KEYS:
            v = sm.get(k); r[k] = f"{v['mean']:.3f} ± {v['std']:.3f}" if v else ""
        r["degenerate"] = f"{sm['degenerate_folds']}/{sm['n_folds']}"
        rows.append(r)
table = pd.DataFrame(rows)
table.to_csv(f"{RESULTS}/results_table.csv", index=False)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 250)
table

,run,balanced_accuracy,auroc,macro_f1,worst_culture_bal_acc,bias_gap_bal_acc,equalized_odds_gap,tpr_gap,fpr_gap,ece,soft_jsd,probe_z_mlp_acc,degenerate
0,openclip_b32_cdd,0.580 ± 0.037,0.623 ± 0.046,0.556 ± 0.059,0.504 ± 0.044,0.134 ± 0.049,0.266 ± 0.078,0.208 ± 0.090,0.245 ± 0.080,0.123 ± 0.044,0.146 ± 0.012,0.983 ± 0.010,0/25
1,openclip_b32_cdd +pcthr,0.570 ± 0.035,0.623 ± 0.046,0.562 ± 0.038,0.506 ± 0.031,0.132 ± 0.051,0.536 ± 0.209,0.506 ± 0.190,0.466 ± 0.232,0.123 ± 0.044,0.146 ± 0.012,0.983 ± 0.010,0/25
2,openclip_b32_cond_soft,0.589 ± 0.035,0.626 ± 0.043,0.576 ± 0.048,0.510 ± 0.047,0.142 ± 0.046,0.338 ± 0.086,0.254 ± 0.096,0.314 ± 0.100,0.144 ± 0.057,0.154 ± 0.019,0.985 ± 0.007,0/25
3,openclip_b32_cond_soft +pcthr,0.577 ± 0.034,0.626 ± 0.043,0.572 ± 0.036,0.513 ± 0.031,0.130 ± 0.042,0.530 ± 0.181,0.447 ± 0.183,0.490 ± 0.192,0.144 ± 0.057,0.154 ± 0.019,0.985 ± 0.007,0/25
4,openclip_b32_cond_soft_dro,0.578 ± 0.036,0.616 ± 0.046,0.561 ± 0.051,0.508 ± 0.049,0.131 ± 0.036,0.280 ± 0.085,0.216 ± 0.105,0.246 ± 0.088,0.147 ± 0.044,0.152 ± 0.014,0.984 ± 0.010,0/25
5,openclip_b32_cond_soft_dro +pcthr,0.569 ± 0.039,0.616 ± 0.046,0.559 ± 0.045,0.500 ± 0.039,0.135 ± 0.049,0.568 ± 0.170,0.520 ± 0.161,0.517 ± 0.185,0.147 ± 0.044,0.152 ± 0.014,0.984 ± 0.010,0/25
6,openclip_b32_leace_cdd,0.579 ± 0.029,0.619 ± 0.040,0.560 ± 0.046,0.509 ± 0.034,0.131 ± 0.039,0.242 ± 0.081,0.179 ± 0.077,0.225 ± 0.081,0.136 ± 0.053,0.151 ± 0.015,0.230 ± 0.035,0/25
7,openclip_b32_leace_cdd +pcthr,0.570 ± 0.022,0.619 ± 0.040,0.560 ± 0.031,0.500 ± 0.028,0.143 ± 0.049,0.552 ± 0.173,0.492 ± 0.182,0.495 ± 0.210,0.136 ± 0.053,0.151 ± 0.015,0.230 ± 0.035,0/25
8,openclip_b32_leace_cdd_dro,0.573 ± 0.038,0.615 ± 0.041,0.550 ± 0.053,0.506 ± 0.041,0.129 ± 0.041,0.209 ± 0.069,0.158 ± 0.077,0.196 ± 0.066,0.131 ± 0.042,0.149 ± 0.013,0.233 ± 0.028,0/25
9,openclip_b32_leace_cdd_dro +pcthr,0.570 ± 0.027,0.615 ± 0.041,0.561 ± 0.032,0.509 ± 0.032,0.128 ± 0.038,0.584 ± 0.174,0.532 ± 0.182,0.529 ± 0.191,0.131 ± 0.042,0.149 ± 0.013,0.233 ± 0.028,0/25


### Corrected significance tests (Nadeau–Bengio resampled t-test)

In [11]:
from scipy import stats

def fold_metric(run, k, part="model"):
    fm = json.load(open(f"{RESULTS}/{run}/fold_metrics.json"))[part]
    return np.array([np.nan if r.get(k) is None else r[k] for r in fm], dtype=float)

def nb_test(x, y, test_train_ratio=0.25):
    """Nadeau & Bengio (2003) corrected resampled t-test for repeated k-fold CV."""
    d = x - y; d = d[~np.isnan(d)]; n = len(d)
    v = d.var(ddof=1)
    if v == 0:
        return d.mean(), 1.0
    t = d.mean() / np.sqrt((1 / n + test_train_ratio) * v)
    return d.mean(), 2 * stats.t.sf(abs(t), n - 1)

TEST_KEYS = ["balanced_accuracy", "auroc", "worst_culture_bal_acc", "equalized_odds_gap", "tpr_gap", "ece", "soft_jsd"]
COMPARE = [("cdd", "plain"), ("cdd", "cond_soft"), ("cond_soft_dro", "cond_soft"),
           ("leace_cdd", "cdd"), ("leace_cdd_dro", "plain"), ("leace_cdd_dro", "cdd")]
out_rows = []
for bk in ["siglip2", "openclip_b32"]:
    for a_, b_ in COMPARE:
        A, B = f"{bk}_{a_}", f"{bk}_{b_}"
        if not (os.path.exists(f"{RESULTS}/{A}/fold_metrics.json") and os.path.exists(f"{RESULTS}/{B}/fold_metrics.json")):
            continue
        r = {"backbone": bk, "comparison": f"{a_} vs {b_}"}
        for k in TEST_KEYS:
            dm, p = nb_test(fold_metric(A, k), fold_metric(B, k))
            r[k] = f"{dm:+.3f} (p={p:.3f})"
        out_rows.append(r)
    # per-culture thresholds vs global threshold for the full model
    A = f"{bk}_leace_cdd_dro"
    if os.path.exists(f"{RESULTS}/{A}/fold_metrics.json"):
        r = {"backbone": bk, "comparison": "leace_cdd_dro: +pcthr vs global thr"}
        for k in TEST_KEYS:
            dm, p = nb_test(fold_metric(A, k, "model_pcthr"), fold_metric(A, k, "model"))
            r[k] = f"{dm:+.3f} (p={p:.3f})"
        out_rows.append(r)
sig = pd.DataFrame(out_rows)
sig.to_csv(f"{RESULTS}/significance_tests.csv", index=False)
sig

,backbone,comparison,balanced_accuracy,auroc,worst_culture_bal_acc,equalized_odds_gap,tpr_gap,ece,soft_jsd
0,siglip2,cdd vs plain,+0.009 (p=0.576),+0.005 (p=0.713),+0.014 (p=0.517),-0.028 (p=0.485),-0.059 (p=0.124),-0.031 (p=0.406),-0.013 (p=0.440)
1,siglip2,cdd vs cond_soft,+0.004 (p=0.776),-0.001 (p=0.946),+0.008 (p=0.748),-0.034 (p=0.304),-0.022 (p=0.563),-0.010 (p=0.435),-0.003 (p=0.474)
2,siglip2,cond_soft_dro vs cond_soft,+0.007 (p=0.648),+0.002 (p=0.790),+0.011 (p=0.568),-0.037 (p=0.463),-0.025 (p=0.616),-0.003 (p=0.836),-0.002 (p=0.691)
3,siglip2,leace_cdd vs cdd,-0.006 (p=0.572),+0.000 (p=0.978),-0.005 (p=0.782),-0.036 (p=0.297),-0.019 (p=0.597),-0.001 (p=0.948),+0.000 (p=0.952)
4,siglip2,leace_cdd_dro vs plain,+0.005 (p=0.744),+0.005 (p=0.606),+0.018 (p=0.283),-0.066 (p=0.214),-0.091 (p=0.088),-0.022 (p=0.576),-0.012 (p=0.448)
5,siglip2,leace_cdd_dro vs cdd,-0.004 (p=0.782),+0.000 (p=0.956),+0.004 (p=0.853),-0.038 (p=0.352),-0.032 (p=0.481),+0.009 (p=0.555),+0.001 (p=0.683)
6,siglip2,leace_cdd_dro: +pcthr vs global thr,-0.011 (p=0.226),+0.000 (p=1.000),-0.020 (p=0.265),+0.251 (p=0.009),+0.242 (p=0.009),+0.000 (p=1.000),+0.000 (p=1.000)
7,openclip_b32,cdd vs plain,+0.007 (p=0.697),+0.013 (p=0.356),+0.007 (p=0.762),-0.086 (p=0.112),-0.064 (p=0.300),-0.035 (p=0.405),-0.017 (p=0.287)
8,openclip_b32,cdd vs cond_soft,-0.010 (p=0.558),-0.003 (p=0.829),-0.006 (p=0.776),-0.072 (p=0.147),-0.046 (p=0.476),-0.021 (p=0.307),-0.008 (p=0.309)
9,openclip_b32,cond_soft_dro vs cond_soft,-0.011 (p=0.421),-0.010 (p=0.182),-0.002 (p=0.927),-0.059 (p=0.214),-0.038 (p=0.541),+0.003 (p=0.869),-0.001 (p=0.810)


### Disagreement alignment: does the model capture *which* memes cultures disagree on?

In [12]:
from scipy.stats import spearmanr
rows = []
for f in sorted(glob.glob(f"{RESULTS}/*/predictions.csv")):
    run = f.split("/")[-2]
    p = pd.read_csv(f)
    rhos = []
    for rep, g in p.groupby("repeat"):          # each meme appears once per repeat
        agg = g.groupby("meme_id").agg(pred_sd=("y_prob", "std"), human_sd=("y_soft", "std"))
        rhos.append(spearmanr(agg["pred_sd"], agg["human_sd"]).correlation)
    rows.append({"run": run, "spearman_rho_mean": np.mean(rhos), "spearman_rho_sd": np.std(rhos, ddof=1)})
align = pd.DataFrame(rows).round(3)
align.to_csv(f"{RESULTS}/disagreement_alignment.csv", index=False)
print("Spearman correlation between predicted and human cross-cultural disagreement per meme")
print("(computed per CV repeat, mean ± sd over repeats; 0 = no alignment)")
align

Spearman correlation between predicted and human cross-cultural disagreement per meme
(computed per CV repeat, mean ± sd over repeats; 0 = no alignment)


,run,spearman_rho_mean,spearman_rho_sd
0,openclip_b32_cdd,0.069,0.046
1,openclip_b32_cond_soft,0.057,0.044
2,openclip_b32_cond_soft_dro,0.074,0.039
3,openclip_b32_leace_cdd,0.039,0.037
4,openclip_b32_leace_cdd_dro,0.068,0.037
5,openclip_b32_plain,0.058,0.046
6,siglip2_cdd,0.181,0.049
7,siglip2_cond_soft,0.181,0.045
8,siglip2_cond_soft_dro,0.180,0.048
9,siglip2_leace_cdd,0.181,0.037


In [13]:
# Most culturally contested memes according to the CDD model (SigLIP 2), for the qualitative analysis section
f = f"{RESULTS}/siglip2_cdd/predictions.csv"
if os.path.exists(f):
    p = pd.read_csv(f)
    g = p.groupby(["meme_id", "culture"]).agg(dev=("deviation_logit", "mean"), cons=("consensus_prob", "mean"),
                                               human=("y_soft", "first")).reset_index()
    spread = g.groupby("meme_id").agg(dev_spread=("dev", lambda v: v.max() - v.min()),
                                      consensus=("cons", "mean"), human_sd=("human", "std"))
    top = spread.sort_values("dev_spread", ascending=False).head(15)
    top.round(3).to_csv(f"{RESULTS}/most_contested_memes.csv")
    print(top.round(3))

         dev_spread  consensus  human_sd
meme_id                                 
57            0.321      0.303     0.347
212           0.317      0.376     0.307
108           0.315      0.557     0.439
264           0.297      0.535     0.319
233           0.287      0.303     0.314
246           0.284      0.551     0.307
10            0.283      0.291     0.268
299           0.283      0.378     0.261
216           0.283      0.449     0.316
98            0.282      0.378     0.460
42            0.276      0.541     0.172
129           0.275      0.669     0.157
29            0.272      0.496     0.260
261           0.270      0.411     0.219
213           0.269      0.657     0.301


In [14]:
import shutil
shutil.make_archive("/kaggle/working/step5_results", "zip", RESULTS)
print("Download /kaggle/working/step5_results.zip from the Output tab.")

Download /kaggle/working/step5_results.zip from the Output tab.


## What to send back
Upload **`step5_results.zip`**. It contains `results_table.csv`, `significance_tests.csv`, `disagreement_alignment.csv`, `most_contested_memes.csv` and all per-fold predictions.